### This notebook contains steps to deploy model as endpoint for inference

In [1]:
!pip install torchmetrics==0.11.4 torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchtext==0.14.1 torchaudio==0.13.1 torchdata==0.5.1 --extra-index-url https://download.pytorch.org/whl/cu117 --no-cache-dir

Looking in indexes: https://pypi.org/simple, https://pip.repos.neuron.amazonaws.com, https://download.pytorch.org/whl/cu117
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 kB 11.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 114.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 204.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 197.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 53.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 242.2 MB/s eta 0:00:00


In [2]:
!pip install h5py --quiet
!pip install scikit-image --quiet
!pip install boto3 --quiet
!pip install "numpy<1.24" --quiet
!pip install skm-tea -q
!pip install 'meddlr[all]' -q
!pip install intensity-normalization --quiet

In [3]:
# ensure torchmetrics version is 0.11.4
import torch
import torchmetrics
print(torch.__version__)
print(torchmetrics.__version__)

1.13.1+cu117
0.11.4


In [9]:
!pip list --format=freeze > requirements.txt

In [4]:
import os
import random
import shutil
import time
import warnings
import pandas as pd
import numpy as np
import h5py
import matplotlib.pyplot as plt
from skimage.color import label2rgb
import torch
from tqdm.notebook import tqdm
from sklearn.model_selection import GroupKFold, KFold
import torch.nn as nn
import copy
import boto3
import json
from io import BytesIO
from io import StringIO
from urllib.parse import urlparse
import math
import skm_tea as st
from meddlr.data.data_utils import collect_mask
import timeit
import utilities
import constants
import os
import re
from sagemaker import get_execution_role, session
from sagemaker.s3 import S3Downloader, S3Uploader

region = boto3.Session().region_name
role = get_execution_role()
sm_session = session.Session(boto3.Session())
sm = boto3.Session().client("sagemaker")
sm_runtime = boto3.Session().client("sagemaker-runtime")

bucket = sm_session.default_bucket()
prefix = "sagemaker/DEMO-segmentation"

Matplotlib is building the font cache; this may take a moment.
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/skm_tea/data/register.py:418: UserWarning: SKM-TEA dataset was not properly registered. Please check that the dataset was properly downloaded and paths have been set. The error is shown below:
[Errno 2] No such file or directory: './datasets/skm-tea/v1-release/all_metadata.csv'
  warnings.warn(


In [15]:
import tarfile

if os.path.exists('model.tar.gz'):
    os.remove('model.tar.gz')
    
# Create 'model.tar.gz' and add 'model.pth', along with other files (e.g., inference.py)
with tarfile.open('model.tar.gz', 'w:gz') as tar:
    # tar.add('model.pth', arcname='model.pth') removing this step as we are using google download
    tar.add('inference.py', arcname='code/inference.py')
    tar.add('constants.py', arcname='code/constants.py')
    tar.add('utilities.py', arcname='code/utilities.py')
    tar.add('requirements.txt', arcname='code/requirements.txt')
    # Add other files/directories as needed, relative to the root directory
    # this model reads the uploaded h5 file. this model does not need any preprocessing 

In [16]:
#upload model.tar.gz to S3
model_url = S3Uploader.upload(
    local_path="model.tar.gz", desired_s3_uri=f"s3://{bucket}/{prefix}"
)

model_url

's3://sagemaker-us-west-1-468914279846/sagemaker/DEMO-segmentation/model.tar.gz'

In [17]:
from sagemaker.image_uris import retrieve
from datetime import datetime

model_name = f"DEMO-seg-prediction"
image_uri = retrieve("pytorch",
                     boto3.Session().region_name, version="1.13.1",
    py_version="py39",image_scope="inference", instance_type="ml.g4dn.4xlarge")
sm_session.create_model(
    name=model_name, role=role, container_defs={"Image": image_uri, "ModelDataUrl": model_url}
)

'DEMO-seg-prediction'

In [18]:
from sagemaker.session import production_variant

variant1 = production_variant(
    model_name=model_name,
    instance_type="ml.g4dn.8xlarge",
    initial_instance_count=1,
    variant_name="Variant1",
    initial_weight=1,
)

(variant1)

{'ModelName': 'DEMO-seg-prediction',
 'VariantName': 'Variant1',
 'InitialVariantWeight': 1,
 'InitialInstanceCount': 1,
 'InstanceType': 'ml.g4dn.8xlarge'}

In [19]:
endpoint_name = f"DEMO-seg-prediction-endpoint"
print(f"EndpointName={endpoint_name}")

sm_session.endpoint_from_production_variants(
    name=endpoint_name, production_variants=[variant1]
)

EndpointName=DEMO-seg-prediction-endpoint
----------!

'DEMO-seg-prediction-endpoint'

In [20]:
# Test endpoint
# Your input data as a Python dictionary
# put slice range : 0-80 and then rerun all below cells again with slice range 80-159
input_data = {'image_id': 'MTR_174', 'slice_start_range':80 , 'slice_end_range':160 }
# Serialize the input data to JSON format
payload = json.dumps(input_data)

# Convert the JSON payload to bytes
payload = payload.encode('utf-8')
sm_runtime.invoke_endpoint(EndpointName=endpoint_name, ContentType="application/json", Body=payload)

{'ResponseMetadata': {'RequestId': 'a34f345b-b3d9-4ab7-b5d6-c94782695bb6',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'a34f345b-b3d9-4ab7-b5d6-c94782695bb6',
   'x-amzn-invoked-production-variant': 'Variant1',
   'date': 'Sat, 29 Jul 2023 20:09:30 GMT',
   'content-type': 'application/json',
   'content-length': '4',
   'connection': 'keep-alive'},
  'RetryAttempts': 0},
 'ContentType': 'application/json',
 'InvokedProductionVariant': 'Variant1',
 'Body': <botocore.response.StreamingBody at 0x7f007d3adff0>}

In [21]:
#check if sliceInfo and summary json is uploaded :: 
# import utilities
# utilities.list_all_objects_in_s3('stats-ui')

In [22]:
#check summary content 
import boto3
import json
def read_json_from_s3(bucket_name, file_key):
    """
    Read a JSON file from the specified AWS S3 bucket.

    Args:
        bucket_name (str): The name of the S3 bucket.
        file_key (str): The key (path) of the JSON file in the S3 bucket.

    Returns:
        dict: The JSON data read from the file.

    Raises:
        botocore.exceptions.ClientError: If the file does not exist or if there's an error reading it.
    """
    s3 = boto3.client('s3')
    response = s3.get_object(Bucket=bucket_name, Key=file_key)
    data = response['Body'].read().decode('utf-8')
    return json.loads(data)

x=read_json_from_s3('stats-ui','segmentation/summary/MTR_174_tissue_summary.json')
print(x)

ClientError: An error occurred (AccessDenied) when calling the GetObject operation: Access Denied